# Notochord Vignette

## Purpose

Explore the dynamics of notochord specification in the context of germ layer restriction.

## Setup

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pycea as py
import scanpy as sc
import seaborn as sns
from devmap.config import (
    celltype_palette,
    germ_layer_palette,
    get_paths,
    set_theme,
    stage_palette,
)
from devmap.fate import annotate_commitment, extant_transition_count
from devmap.plots import plot_restriction_sankey
from devmap.progenitors import mark_sibling_descendants
from devmap.utils import load_data, save_plot

set_theme()
base_path, plots_path, results_path = get_paths("vignettes")

## Load data

Load data

In [ ]:
tdata = load_data("log1p_hvg")
cluster_umaps = pd.read_csv(base_path / "embedding" / "results" /"cluster_umaps.csv", index_col=0)

Load progenitors

In [ ]:
fate_results = base_path / "fate" / "results"
progenitors = pd.read_csv(fate_results / "cell_type_progenitors.csv")
progenitor_assignment = pd.read_csv(fate_results / "cell_type_progenitor_descendants.csv", index_col=0)
germ_layer_progenitors = pd.read_csv(fate_results / "germ_layer_progenitors.csv", index_col=0)
germ_layer_assignment = pd.read_csv(fate_results / "germ_layer_progenitor_descendants.csv", index_col=0)

## UMAPs

In [13]:
cluster = 'early'
cluster_tdata = tdata[cluster_umaps.query("cluster == @cluster").index].copy()
cluster_tdata.obsm["X_umap"] = cluster_umaps.query("cluster == @cluster")[["umap_1", "umap_2"]].values

plot

In [ ]:
for gene in ["Noto","Foxa2"]:
    fig, ax = plt.subplots(figsize=(3,3), dpi=300)
    sc.pl.umap(cluster_tdata, color = gene, ax=ax, show=False, s=2, vmax = 3.5, frameon=False)
    save_plot(plots_path / f"{gene}_notochord_umap.svg", fig, rasterize = True)

## Mark progenitors

In [ ]:
notochord_progenitors = progenitors.query("fate == 'Notochord' and fate_descendants > 1")
mark_sibling_descendants(tdata, notochord_progenitors)
tdata.obs["sibling_prog_time"] = tdata.obs["sibling_descendants"].map(notochord_progenitors.set_index("node")["time"])

## Sankey plot

In [ ]:
e95_tdata = tdata[(tdata.obs.stage == "E9.5") & tdata.obs.clone.notnull()].copy()
e95_tdata.obs["filtered_germ_layer"] = e95_tdata.obs["germ_layer"].replace({"Epiblast": "Other","Primordial germ cell": pd.NA})
e95_tdata.obs.loc[e95_tdata.obs.cell_type == "Notochord", "filtered_germ_layer"] = pd.NA
germ_assignment = e95_tdata.obs["filtered_germ_layer"].to_dict()
notochord_assignment = e95_tdata.obs["cell_type"].map(lambda x: "Notochord" if x == "Notochord" else "Other").to_dict()
for tree in e95_tdata.obst.values():
    annotate_commitment(tree, [germ_assignment, notochord_assignment], threshold=0.9, exclude_categories=["Other"])

In [ ]:
paths = extant_transition_count(e95_tdata[e95_tdata.obs.cell_type == "Notochord"], key = "commitment",
                                depth_key="time", bins = np.arange(0, 10, .5))
fig, ax = plt.subplots(figsize=(2.8, 2),dpi=600)
sankey_palette = sankey_palette = {**germ_layer_palette,**{"Uncommitted": "#CCCCCC"}}
sankey_palette["Notochord"] = celltype_palette["Notochord"]
sankey_order = ["Uncommitted", "Ectoderm","Mesoderm", "Endoderm", "Notochord"]
plot_restriction_sankey(paths.query("source_time > 3"), normalize = True, order = sankey_order, palette = sankey_palette, ax = ax)
plt.xticks([1,5,9,13], ["4", "6", "8", "10"]);
plt.xlim(0, 13)
plt.xticks(rotation=0)
save_plot(plots_path / "notochord_sankey.svg", fig)

## Sibling fates

Germ layer enrichment

In [36]:
df = tdata.obs.query("clone.notnull() and germ_layer != 'Epiblast' and cell_type != 'Notochord'")

background = df["germ_layer"].value_counts(normalize=True)
sibling_pct = df.loc[df["sibling_descendants"].notna(), "germ_layer"].value_counts(normalize=True)

sibling_germ_layer = (
    sibling_pct.mul(100).rename("pct")
    .to_frame()
    .assign(enrichment=sibling_pct.div(background))
    .sort_values("enrichment", ascending=False)
    .reset_index(names="germ_layer")
)

In [ ]:
for stat in ["pct", "enrichment"]:
    fig, ax = plt.subplots(figsize=(.8,.4), dpi=600)
    sns.barplot(data = sibling_germ_layer, x = stat, y = "germ_layer", hue = "germ_layer",
        palette = germ_layer_palette, order = sibling_germ_layer.germ_layer[:3], saturation=1, ax = ax)
    if stat == "enrichment":
        plt.xlim(0,15)
        plt.xticks([0,5,10,15])
    else:
        plt.xlim(0,65)
        plt.xticks([0,20,40,60])
    plt.yticks(fontsize=8)
    plt.ylabel("")
    save_plot(plots_path / f"notochord_germ_layer_sibling_enrichment_{stat}.svg", fig)

Cell type enrichment

In [ ]:
df = tdata.obs.query("stage != 'E7.5' & clone.notnull() & germ_layer != 'Epiblast' & cell_type != 'Notochord'").copy()
type_counts = df.groupby(["cell_type","germ_layer"], observed = True).agg(
    sibling_count = ("sibling_descendants", lambda x: x.notnull().sum()),
    total_count = ("cell_type", "count")
).reset_index()
total_ratio = type_counts["total_count"].sum() / type_counts["sibling_count"].sum()
type_counts["enrichment"] = (type_counts["sibling_count"] / type_counts["total_count"]) * total_ratio


fig, ax = plt.subplots(figsize=(2.5, 2.5), dpi=600)
sns.scatterplot(data = type_counts, x = "total_count", y = "sibling_count", hue = "germ_layer", palette = germ_layer_palette, legend = False, ax = ax, edgecolor = "black", linewidth = 0.5)
plt.plot([100, 80000], [100 / total_ratio, 80000 / total_ratio], color = "black", linestyle = "--", zorder = -1)
plt.xscale("log")
plt.yscale("log")
plt.xlim(50, 5e5)
save_plot(plots_path / "notochord_sibling_enrichment.svg", fig)

## Notochord vs germ layer specification timing

In [ ]:
tdata.obs["notochord_progenitor"] = progenitor_assignment.query("fate == 'Notochord'")["progenitor"]
tdata.obs["notochord_progenitor_time"] = tdata.obs["notochord_progenitor"].map(notochord_progenitors.set_index("node")["time"])
tdata.obs["germ_layer_progenitor"] = germ_layer_assignment["progenitor"]
tdata.obs["germ_layer_progenitor_time"] = tdata.obs["germ_layer_progenitor"].map(germ_layer_progenitors.groupby("node")["time"].first())
tdata.obs["time_delta"] = tdata.obs["notochord_progenitor_time"] - tdata.obs["germ_layer_progenitor_time"]
df = tdata.obs.groupby(["notochord_progenitor"]).agg(
    {"germ_layer_progenitor_time":"first", "notochord_progenitor_time":"first",
    "time_delta":"first", "cell_type":"count","stage":"first"}).query("stage != 'E10.0'")

In [ ]:
fig = plt.figure(figsize=(2.2, 1.8), dpi=600)
gs = fig.add_gridspec(1, 2, width_ratios=(4, 1), wspace=0.05)

ax = fig.add_subplot(gs[0, 0])
ax_marg_y = fig.add_subplot(gs[0, 1], sharey=ax)

sns.scatterplot(
    data=df,
    x="germ_layer_progenitor_time",
    y="notochord_progenitor_time",
    hue="stage",
    palette=stage_palette,
    s=5,
    edgecolor="black",
    legend=False,
    ax=ax,
)

ax.plot([4, 9.5], [4, 9.5], color="black", linestyle="--", linewidth=1, zorder=-1)

sns.kdeplot(
    data=df,
    y="notochord_progenitor_time",
    hue="stage",
    bw_adjust=1.2,
    palette=stage_palette,
    legend=False,
    common_norm=True,
    ax=ax_marg_y,
)

mean_time = df["notochord_progenitor_time"].mean()
print(mean_time)
ax_marg_y.axhline(mean_time, color="k", linestyle="--", linewidth=1, zorder=-1)

ax.set_xticks([4, 6, 8, 10])
ax.set_yticks([4, 6, 8, 10])

ax_marg_y.set_xticks([])
ax_marg_y.set_ylabel("")
ax_marg_y.tick_params(axis="y", left=False, labelleft=False)

save_plot(plots_path / "notochord_vs_germ_layer_timing.svg", fig, rasterize=True)

## Fate-restricted clades

tree

In [ ]:
fig, ax = plt.subplots(figsize=(1.2, 3.6), dpi = 600)
clone = 'E9.5-R2-C1'
nodes = notochord_progenitors.query("clone == @clone")["node"].tolist()
py.pl.tree(e95_tdata, depth_key="time", tree = clone, ax = ax, branch_linewidth=.5)
py.pl.nodes(e95_tdata,tree = clone, nodes = nodes, ax = ax, color = celltype_palette["Notochord"], size = 15, outline_width=.3)
save_plot(plots_path / "notochord_progenitor_tree.svg", fig, rasterize=True)


example clades

In [ ]:
example_nodes = ["4W2KBATo", "T0ELzNgn","5f6YKBiX"]
for node in example_nodes:
    fig, ax = plt.subplots(figsize=(2,.3), dpi=600)
    subset = e95_tdata[e95_tdata.obs.sibling_descendants == node].copy()
    py.pl.tree(subset, depth_key="time", ax = ax, branch_linewidth=0.5)
    py.pl.annotation(subset, keys = "germ_layer", palette = germ_layer_palette, label = False, ax = ax, gap = .01, legend = False)
    py.pl.annotation(subset, keys = "cell_type", palette = celltype_palette, label = False, ax = ax, gap = .01, legend = True)
    save_plot(plots_path / f"notochord_prog_tree_{node}.svg", fig)